# 情報数学Ⅲ 第15回

In [ ]:
# 必要なデータファイルを取得
import requests

base_url = "https://raw.githubusercontent.com/logics-of-blue/book-python-stats-2nd/refs/heads/main/book-data/"
filenames = [
    "9-2-1-logistic-regression.csv"
]

for filename in filenames:
    url = base_url + filename
    print(f"Downloading {filename}...")
    response = requests.get(url)
    if response.status_code == 200:
        with open(filename, "wb") as f:
            f.write(response.content)
    else:
        print(f"Failed to download {filename}: {response.status_code}")

# 第9部　一般化線形モデル

## 2章　ロジスティック回帰

### 実装：分析の準備

In [ ]:
# 数値計算に使うライブラリ
import numpy as np
import pandas as pd
from scipy import stats
# 表示桁数の設定
pd.set_option('display.precision', 3)
np.set_printoptions(precision=3)

# グラフを描画するライブラリ
from matplotlib import pyplot as plt
import seaborn as sns
sns.set()

# 統計モデルを推定するライブラリ
import statsmodels.formula.api as smf
import statsmodels.api as sm

In [ ]:
# 表示設定(書籍本文のレイアウトと合わせるためであり、必須ではありません)
np.set_printoptions(linewidth=60)
pd.set_option('display.width', 60)

from matplotlib.pylab import rcParams
rcParams['figure.figsize'] = 8, 4

### 実装：データの読み込みと可視化

In [ ]:
# データの読み込み
test_result = pd.read_csv('9-2-1-logistic-regression.csv')
print(test_result.head(3))

In [ ]:
# データの図示
sns.barplot(x='hours',y='result', 
            data=test_result, palette='gray_r')

In [ ]:
# 勉強時間ごとの合格率
print(test_result.groupby('hours').mean())

### 実装：ロジスティック回帰

In [ ]:
# モデル化
mod_glm = smf.glm(formula='result ~ hours', 
                  data=test_result, 
                  family=sm.families.Binomial()).fit()

In [ ]:
# 参考：リンク関数を指定する(書籍には載っていないコードです)
logistic_reg = smf.glm(formula = 'result ~ hours', 
                       data = test_result, 
                       family=sm.families.Binomial(link=sm.families.links.Logit())).fit() # logit→Logitに変更 by ggszk

### 実装：ロジスティック回帰の結果の出力

In [ ]:
# 結果の出力
mod_glm.summary()

### 実装：ロジスティック回帰のモデル選択

In [ ]:
# Nullモデル
mod_glm_null = smf.glm(
    'result ~ 1', data=test_result, 
    family=sm.families.Binomial()).fit()

In [ ]:
# AICの比較
print('Nullモデル　　：', round(mod_glm_null.aic, 3))
print('変数入りモデル：', round(mod_glm.aic, 3))

### 実装：ロジスティック回帰による予測

#### predict関数を使った予測

In [ ]:
# 0~9まで1ずつ増える等差数列
exp_val = pd.DataFrame({
    'hours': np.arange(0, 10, 1)
})
# 成功確率の予測値
pred = mod_glm.predict(exp_val)
pred

#### 推定された係数を使った予測

In [ ]:
beta0 = mod_glm.params[0]
beta1 = mod_glm.params[1]
hour = 9

round(1 / (1 + np.exp(-(beta0 + beta1 * hour))), 3)

### 実装：ロジスティック回帰の回帰曲線の図示

In [ ]:
# lmplotでロジスティック回帰曲線を図示する
sns.lmplot(x='hours', y='result',
           data=test_result, logistic=True,
           scatter_kws = {'color': 'black'},
           line_kws    = {'color': 'black'},
           x_jitter=0.1, y_jitter=0.02,
           ci=None, height=4, aspect=2)

### 実装：ロジスティック回帰の係数とオッズ比の関係

In [ ]:
# 勉強時間が1時間である場合の合格率
exp_val_1 = pd.DataFrame({'hours': [1]})
pred_1 = mod_glm.predict(exp_val_1)

# 勉強時間が2時間である場合の合格率
exp_val_2 = pd.DataFrame({'hours': [2]})
pred_2 = mod_glm.predict(exp_val_2)

In [ ]:
# オッズ
odds_1 = pred_1 / (1 - pred_1)
odds_2 = pred_2 / (1 - pred_2)

# 対数オッズ比
log_odds_ratio = np.log(odds_2 / odds_1)
log_odds_ratio

In [ ]:
# 係数
round(mod_glm.params['hours'], 3)

In [ ]:
# 補足：オッズ比に戻す
round(np.exp(mod_glm.params['hours']), 3)